In [1]:
#Example-1
from langchain_aws import ChatBedrockConverse 
from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.runnables import RunnablePassthrough

In [2]:
llm = ChatBedrockConverse(model_id = "cohere.command-r-plus-v1:0", region_name = 'us-east-1', temperature = 0.5, max_tokens = 550) 

In [6]:
knowledge_prompt = ChatPromptTemplate.from_template("Generate 3-5 essential common-sense facts related to the physics or logic required to answer this question: {question}") 
answer_prompt = ChatPromptTemplate.from_template("""
                KNOWLEDGE: 
                {knowledge}
                
                QUESTION: 
                {question} 
                
                Using the knowledge provided above, provide a logical answer to the question.
                """)

In [7]:
knowledge_chain = knowledge_prompt | llm | StrOutputParser() 
gkp_chain = {'knowledge':knowledge_chain, 'question': RunnablePassthrough()}| answer_prompt | llm | StrOutputParser()
            

In [8]:
gkp_chain.invoke({'question':'Would a person use a toaster while taking a bath?'})

"No, a person should not use a toaster while taking a bath. \n\nUsing electrical appliances like a toaster in close proximity to a bathtub full of water is extremely dangerous and could lead to serious accidents, including electrical shocks or fires. This is because water is an excellent conductor of electricity, and any contact with water can result in electricity taking the path of least resistance, potentially through the person in the tub, resulting in electrocution. \n\nAdditionally, bathrooms are high-moisture environments with steam and water condensation, which can affect electrical appliances. Toasters are designed for dry environments, and the moisture in the bathroom could affect the toaster's functionality, leading to a potential malfunction. \n\nTherefore, it is important to keep electrical appliances away from the bathtub and only use them in dry areas, following basic electrical safety guidelines to prevent hazardous situations."

In [11]:
print(knowledge_chain.invoke({'question':'Would a person use a toaster while taking a bath?'}))

Sure! Here are some essential common-sense facts related to the physics and logic of the situation: 

1. Electrical appliances and water do not mix: Toasters, like most electrical appliances, are not designed to be used in close proximity to water. Water is a conductor of electricity, and if a toaster were to come into contact with water, it could result in electrical shock, short circuits, or even start a fire. 

2. Toasters are designed for kitchen counters: Toasters are typically designed with stability and countertop use in mind. They have non-slip bases and are meant to be used on stable, flat surfaces. The uneven and potentially slippery surface of a bathtub would not provide a safe or practical place to use a toaster. 

3. Bathing and electricity don't mix: Bathing, especially in a bathtub filled with water, is an activity that should be free of electrical hazards. The risk of electrical shock is heightened when immersed in water, and any electrical appliance brought into the ba

In [1]:
#Example -2 
from getpass import getpass
from langchain_core.prompts import PromptTemplate
import os

In [2]:
template = """Question: {question} Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

In [3]:
from langchain_aws import ChatBedrockConverse
llm1=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '2f85cac3-0539-4607-bb4a-e576c63a51ea', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 01:56:07 GMT', 'content-type': 'application/json', 'content-length': '232', 'connection': 'keep-alive', 'x-amzn-requestid': '2f85cac3-0539-4607-bb4a-e576c63a51ea'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [428]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'}, id='lc_run--019d9400-b074-7160-82d4-f9d202aca548-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1, 'output_tokens': 9, 'total_tokens': 10, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

In [4]:
template1 = "You are a writer. You are Given with the name of a Bird, it is your job to write a brief note for that Name.\
             Name: {name}   BriefNote: This is a briefnote for the above bird:"
prompt1 = PromptTemplate(template=template1, input_variables=["name"])
Chain_one = prompt1 | llm1

In [5]:
llm2=ChatBedrockConverse(model="amazon.nova-lite-v1:0",
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
template2 = "You are an expert in zoology. Given a brief note about a bird, it is your job to write a review comments for that Bird.\
             briefNote: {note}   BriefNote: Review for a briefnote on bird: "
prompt2 = PromptTemplate(template=template2, input_variables=["note"])
Chain_two = prompt2 | llm2 



In [6]:
note=Chain_one.invoke("peacock").content
print(note)

The peacock is a stunning and majestic bird, known for its vibrant plumage and exquisite beauty. With a colorful fan of feathers, it captivates all who witness its graceful strut and elegant display. A true symbol of splendor, the peacock embodies the essence of natural artistry and never fails to inspire awe and admiration.


In [7]:
print(Chain_two.invoke(note).content)

**Review Comments:**

The peacock is indeed a breathtaking and awe-inspiring bird, renowned for its vivid and ornate plumage. Its vivid, iridescent feathers create a mesmerizing fan that not only captivates the eye but also showcases the incredible artistry found in nature. The peacock's graceful strut and elegant display are a testament to its majestic presence. This magnificent bird stands as a true symbol of splendor and beauty, effortlessly inspiring admiration and wonder in all who have the privilege of witnessing its natural elegance. The peacock's allure is undeniable, making it a timeless icon in the avian world.


In [9]:
def log_note_function(x): 
    print("Note on bird:",x) 
    return x 

log_note = RunnableLambda(log_note_function)

In [10]:
from langchain_core.runnables import RunnableLambda
# Define a logging step to show the intermediate output
#log_note = RunnableLambda(lambda x: print("📝 Note on bird:", x) or x)  # Logs intermediate result and Passes the same data to the next chain step
# Now combine the full chain
combined_chain = Chain_one | log_note | Chain_two
# Invoke the full chain
final_output = combined_chain.invoke({"name": "bird of paradise"})
# Show final result
print("___"*45,"✅ Final output:", final_output)


Note on bird: content='The bird of paradise is an exquisite creature, a true embodiment of natural artistry. With vibrant plumage that seems to dance in the breeze, this bird captivates all who lay eyes on it. Its name befits its majestic presence, bringing to mind tropical havens and untamed beauty. This bird is a reminder of the wonders that nature holds, a colorful spectacle that leaves an everlasting impression.' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '53420b4b-704a-4a1b-ae7a-e380992e4b90', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 16 Apr 2026 01:58:28 GMT', 'content-type': 'application/json', 'content-length': '600', 'connection': 'keep-alive', 'x-amzn-requestid': '53420b4b-704a-4a1b-ae7a-e380992e4b90'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [2125]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019d9402-d240-7001-921c-015848457e4e-0' tool_calls=[] invali